# Lab — MLflow: from a notebook run to a served model

Start a tracking server, log runs, package a model as `pyfunc`, check what goes in and out,
compare experiments, register and serve the chosen one.

Everything runs inside this Colab VM: the tracking server, its database, and later the model
server. Nothing leaves the machine, and everything is gone when the runtime is.

| Part | What you do | Minutes |
|---|---|---|
| 0 Setup | start the tracking server, open the UI | 5 |
| 1 Data | load wine quality, split | 5 |
| 2 A pyfunc model | **exercise**: put encoding, model and threshold in one class | 15 |
| 3 A first run | **exercise**: log what you need to reuse the run in six months | 15 |
| 4 The contract | send bad rows to the model, see what gets through | 10 |
| 5 Experiments | **exercise**: a function that runs and logs one experiment; compare runs | 20 |
| 6 Registry | **exercise**: promote a model and roll it back | 10 |
| 7 Serve | **exercise**: call the model over HTTP | 10 |

**How to work.** Cells marked *Plumbing* are given: run them and move on. Each exercise is a
stub to fill in, followed in the same cell by a call to a test from `mlflow_tests.py`. The test
prints that it passed, or stops with a line saying what is wrong. Under each exercise there is a
folded solution: try first, and open it if you are stuck for more than a few minutes.

## Part 0 · Setup

In [ ]:
# Plumbing: run and skip. MLflow version pinned to the one this lab was tested with.
%pip install -q mlflow==3.18.0

In [ ]:
# Plumbing: run and skip. Imports.
import io
import logging
import os
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore", module="mlflow")
for noisy in ["mlflow.pyfunc", "mlflow.utils.environment", "mlflow.tracking.fluent", "mlflow.tracking._model_registry.fluent",
              "mlflow.store.model_registry.abstract_store"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)

import mlflow
import mlflow.pyfunc
import pandas as pd
import requests
from mlflow.models import infer_signature

In [ ]:
# Plumbing: run and skip. Gets the tests file, mlflow_tests.py, next to this notebook.
TESTS_URL = "https://raw.githubusercontent.com/vigji/applied-ai-labs/main/day-0/mlflow/mlflow_tests.py"
TESTS = Path("mlflow_tests.py")

if not TESTS.exists():
    source = os.environ.get("LAB_TESTS") or TESTS_URL
    if not source:
        raise RuntimeError("No tests file. Put the link to mlflow_tests.py into TESTS_URL above, then run this cell again.")
    if Path(source).exists():
        TESTS.write_text(Path(source).read_text())
    elif "drive.google.com" in source:
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
        import inspect
        fuzzy = {"fuzzy": True} if "fuzzy" in inspect.signature(gdown.download).parameters else {}   # gdown 6 dropped this argument
        gdown.download(source, str(TESTS), quiet=True, **fuzzy)
    else:
        urllib.request.urlretrieve(source, TESTS)
    if not TESTS.exists() or "def test_features" not in TESTS.read_text():
        TESTS.unlink(missing_ok=True)
        raise RuntimeError("The file that was downloaded is not mlflow_tests.py. Check the link in TESTS_URL, "
                           "then run this cell again.")

import mlflow_tests as tests

The tracking server is a small web application with a database behind it. Your code sends it
parameters, metrics and files; the UI reads them back. Here it runs in the background of this VM,
with its records in `mlflow.db` and the files (models, plots) in `./mlartifacts`. In a team it
would run on a shared machine, and everyone would point `set_tracking_uri` at it.

In [ ]:
# Plumbing: run and skip. Tracking server and model registry, in the background (up to two minutes on Colab).
# --allowed-hosts and --cors-allowed-origins let the UI through Colab's proxy, which calls the
# server with the VM's internal name; MLflow refuses unknown names by default.
PORT = 5000
TRACKING_URI = f"http://127.0.0.1:{PORT}"

server = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(PORT),
     "--backend-store-uri", "sqlite:///mlflow.db", "--artifacts-destination", "./mlartifacts",
     "--allowed-hosts", "*.codatalab-user-runtimes.internal:*,localhost:*,127.0.0.1:*", "--cors-allowed-origins", "https://*.colab.dev"],
    stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(180):
    try:
        if requests.get(f"{TRACKING_URI}/health").ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("server did not start, see mlflow_server.log")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("wine")

In [ ]:
# Plumbing: run and skip. The link to the MLflow UI. Open it in a new tab and keep it open.
# In the UI, switch the toggle at the top left from "GenAI" to "Model training".
try:
    from google.colab.output import eval_js
    print(eval_js(f"google.colab.kernel.proxyPort({PORT})"))
except ImportError:
    print(TRACKING_URI)

## Part 1 · Data

Wine quality (UCI). Red and white wines, physico-chemical measurements, a quality score from 3 to 9.
Target: `good` = quality >= 7.

About one wine in five is good. A model that always says "not good" is right four times out of
five, so accuracy alone will flatter every model: read balanced accuracy and ROC AUC next to it.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(DATA_URL).read()))

frames = []
for color in ["red", "white"]:
    df = pd.read_csv(z.open(f"winequality-{color}.csv"), sep=";")
    df.insert(0, "type", color)
    frames.append(df)

wine = pd.concat(frames, ignore_index=True)
wine["good"] = (wine["quality"] >= 7).astype(int)
wine = wine.drop(columns="quality")

print(wine.shape, "positives:", wine["good"].mean().round(3))
wine.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = wine.drop(columns="good")
y = wine["good"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

## Part 2 · A pyfunc model

MLflow can save a scikit-learn estimator as it is. But the estimator is not the whole model:
something has to turn `type` into a number before calling it, and something has to pick the
threshold that turns a probability into a yes or a no. If that code stays in the notebook,
whoever loads the model has to write it again, and will write it slightly differently.

A `pyfunc` model is a Python class with a `predict(context, model_input, params)` method.
MLflow pickles the object and stores it with the list of packages it needs. From then on every
consumer calls the same `predict`: `mlflow.pyfunc.load_model(...)` in Python, or the REST server
in part 7. Whatever happens between a raw row and the prediction lives inside the class and is
logged with it.

### Exercise · implement `WineModel`

> Difficulty: 🔴🔴⚪⚪⚪ · Importance: 🔵🔵🔵🔵⚪ · about 15 minutes

Fill in the three methods. `fit` and `predict` are a few lines each; the point is where the
encoding and the threshold live, not the code. `predict` gets `context` and `params` from MLflow:
you can ignore both here.

In [ ]:
class WineModel(mlflow.pyfunc.PythonModel):

    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator   # any sklearn classifier with predict_proba
        self.threshold = threshold   # p_good at or above this means good

    @staticmethod
    def _features(df: pd.DataFrame) -> pd.DataFrame:
        """Raw rows -> what the estimator takes.

        `type` is "red" or "white": make it red = 1, white = 0 (when the column is there: in part 5
        you will train on fewer columns). Return a new DataFrame and leave `df` unchanged.
        """
        pass  # YOUR CODE HERE

    def fit(self, X: pd.DataFrame, y) -> "WineModel":
        """Fit the estimator on the encoded rows. Return self."""
        pass  # YOUR CODE HERE

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        """Raw rows -> a DataFrame with two columns:
        `p_good`, the estimator's probability of class 1, and `good`, 1 when p_good >= threshold, else 0.
        """
        pass  # YOUR CODE HERE


tests.test_features(WineModel)
tests.test_fit(WineModel)
tests.test_predict(WineModel)

<details><summary>Solution</summary>

```python
class WineModel(mlflow.pyfunc.PythonModel):

    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator   # any sklearn classifier with predict_proba
        self.threshold = threshold   # p_good at or above this means good

    @staticmethod
    def _features(df: pd.DataFrame) -> pd.DataFrame:
        """Raw rows -> what the estimator takes.

        `type` is "red" or "white": make it red = 1, white = 0 (when the column is there: in part 5
        you will train on fewer columns). Return a new DataFrame and leave `df` unchanged.
        """
        df = df.copy()
        if "type" in df:
            df["type"] = (df["type"] == "red").astype(int)
        return df

    def fit(self, X: pd.DataFrame, y) -> "WineModel":
        """Fit the estimator on the encoded rows. Return self."""
        self.estimator.fit(self._features(X), y)
        return self

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        """Raw rows -> a DataFrame with two columns:
        `p_good`, the estimator's probability of class 1, and `good`, 1 when p_good >= threshold, else 0.
        """
        p = self.estimator.predict_proba(self._features(model_input))[:, 1]
        return pd.DataFrame({"good": (p >= self.threshold).astype(int), "p_good": p})


tests.test_features(WineModel)
tests.test_fit(WineModel)
tests.test_predict(WineModel)
```

</details>

In [ ]:
# Plumbing: run and skip. The metrics every run reports.
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, roc_auc_score)


def metrics(y_true, out):
    return {
        "accuracy": accuracy_score(y_true, out["good"]),
        "balanced_accuracy": balanced_accuracy_score(y_true, out["good"]),
        "precision": precision_score(y_true, out["good"], zero_division=0),
        "recall": recall_score(y_true, out["good"]),
        "roc_auc": roc_auc_score(y_true, out["p_good"]),
    }

## Part 3 · A first run

A run is one record on the tracking server. It holds **parameters** (what you chose),
**metrics** (the numbers that came out) and **artifacts** (files, the model among them).
The question to ask when deciding what to log: if someone finds this run in six months,
can they tell what was done, and use the model without asking you?

Two things go with the model:

- the **signature**, `infer_signature(inputs, outputs)`: the names and types of the columns that
  go in and come out. MLflow checks every call against it.
- an **input example**: a few real rows stored next to the model, so that whoever picks it up
  sees what a valid input looks like.

### Exercise · log the run

> Difficulty: 🔴🔴⚪⚪⚪ · Importance: 🔵🔵🔵🔵🔵 · about 15 minutes

Training and scoring are given. Log, inside the `with` block:

- parameters, with `mlflow.log_params(dict)`: the estimator's, and also the threshold and the
  feature list, which change the model as much as `n_estimators` does
- the metrics, with `mlflow.log_metrics(dict)`
- the model, with `mlflow.pyfunc.log_model(name="model", python_model=..., input_example=..., signature=...)`

Anything else you would want in six months? The data's URL, the split seed, the number of rows,
who ran it. Add it with `mlflow.set_tag(key, value)` or `mlflow.set_tags(dict)`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

params = {"n_estimators": 100, "max_depth": None, "random_state": 0}

with mlflow.start_run(run_name="rf-baseline") as run:
    model = WineModel(RandomForestClassifier(**params)).fit(X_train, y_train)
    out = model.predict(None, X_test)
    m = metrics(y_test, out)

    pass  # YOUR CODE HERE

print(run.info.run_id)
tests.test_first_run(run.info.run_id, X_train)
m

<details><summary>Solution</summary>

```python
from sklearn.ensemble import RandomForestClassifier

params = {"n_estimators": 100, "max_depth": None, "random_state": 0}

with mlflow.start_run(run_name="rf-baseline") as run:
    model = WineModel(RandomForestClassifier(**params)).fit(X_train, y_train)
    out = model.predict(None, X_test)
    m = metrics(y_test, out)

    mlflow.log_params({**params, "threshold": model.threshold, "features": list(X_train.columns)})
    mlflow.log_metrics(m)
    mlflow.set_tags({"data": DATA_URL, "split_seed": 0, "n_train": len(X_train), "n_test": len(X_test)})
    mlflow.pyfunc.log_model(
        name="model",
        python_model=model,
        input_example=X_train.head(3),
        signature=infer_signature(X_train, out),
    )

print(run.info.run_id)
tests.test_first_run(run.info.run_id, X_train)
m
```

</details>

Now load the model back from the server, as anyone else would: by the run's id, without
`WineModel` or `X_train` from this notebook's code.

In [ ]:
loaded = mlflow.pyfunc.load_model(f"runs:/{run.info.run_id}/model")
print(loaded.metadata.signature)
loaded.predict(X_test.head())

In the UI (*Model training* mode): open the `wine` experiment and the run. Find its parameters,
metrics and tags. The model has a page of its own: click `model` next to any metric, or open the
experiment's *Models* tab. Its *Artifacts* tab holds the `MLmodel` file (how to load it, the
signature), `requirements.txt` (the packages it needs) and `input_example.json`.

## Part 4 · What goes in, what comes out

The signature is a contract between the model and whoever calls it. Try to break it.

In [ ]:
row = X_test.head(1)

cases = {
    "missing column": row.drop(columns="alcohol"),
    "wrong type": row.assign(alcohol="12.5"),
    "extra column": row.assign(vintage=2019),
    "missing value": row.assign(alcohol=None),
    "unknown category": row.assign(type="rosé"),
    "values out of range": row.assign(alcohol=90.0, pH=-3.0),
}

for name, df in cases.items():
    try:
        result = loaded.predict(df).to_dict(orient="records")
    except Exception as e:
        result = f"{type(e).__name__}: {str(e).rsplit('Error:', 1)[-1].strip()[:200]}"
    print(f"{name:22s} -> {result}")

Which of these should fail and did not? Which failed and should not?
Where does each check belong: in the signature, in `WineModel.predict`, in the caller?
Add your own cases.

<details><summary>Answer</summary>

| case | what happened | should it? | where the check belongs |
|---|---|---|---|
| missing column | refused by the signature | yes | signature |
| wrong type, `"12.5"` as text | a prediction: the text is read as 12.5 | arguable | signature, or the caller |
| extra column | ignored, with a warning | fine | signature |
| missing value | a prediction | no | `predict` |
| unknown category, `rosé` | a prediction | no | `_features` |
| values out of range | a prediction | no | `predict`, or the caller |

These are the results on Colab (pandas 2.2). With pandas 3 the text `"12.5"` is refused instead:
the same model and the same signature, a different answer, because a library underneath changed.

The signature checks names and types, and that required columns are present. It does not
check that a column has a value, which values a text column may take, or what is physically
possible.

- *Missing value*: the random forest accepts NaN (scikit-learn 1.4 and later) and answers.
  Refuse the row in `predict`, or fill it in on purpose; either way, decide.
- *Rosé*: `_features` only asks "is it red?", so a rosé is scored as a white. Only the model
  knows its categories: list them in `_features` and refuse the others.
- *90% alcohol, pH −3*: a valid float, an impossible wine, and the model still gives a
  probability. Ranges are domain knowledge. A check in `predict` travels with the model; a check
  in the caller has to be written again by every caller.

</details>

## Part 5 · Experiments

One run is a record; many runs are a comparison. Every run below goes into the same experiment,
`wine`, and the UI (or `mlflow.search_runs`) puts them side by side. This only works if every run
logs the same things in the same way, which is why it is worth a function.

### Exercise · implement `run_experiment`

> Difficulty: 🔴⚪⚪⚪⚪ · Importance: 🔵🔵🔵⚪⚪ · about 10 minutes

The same as your first run, on `X_train[features]` and `X_test[features]`, with the given
threshold. Log `estimator.get_params()`, the estimator's class name as `estimator`, the threshold
and the feature list. Infer the signature and take the input example on the selected features only.
The test runs your function once and deletes the run it made.

In [ ]:
def run_experiment(estimator, features=None, threshold=0.5, run_name=None, tags=None):
    features = features or list(X_train.columns)
    with mlflow.start_run(run_name=run_name, tags=tags) as run:
        m = None  # YOUR CODE HERE: train, score and log, as in the first run
    return run.info.run_id, m


tests.test_run_experiment(run_experiment)

<details><summary>Solution</summary>

```python
def run_experiment(estimator, features=None, threshold=0.5, run_name=None, tags=None):
    features = features or list(X_train.columns)
    with mlflow.start_run(run_name=run_name, tags=tags) as run:
        model = WineModel(estimator, threshold).fit(X_train[features], y_train)
        out = model.predict(None, X_test[features])
        m = metrics(y_test, out)

        mlflow.log_params({**estimator.get_params(), "estimator": type(estimator).__name__,
                           "threshold": threshold, "features": features})
        mlflow.log_metrics(m)
        mlflow.pyfunc.log_model(
            name="model",
            python_model=model,
            input_example=X_train[features].head(3),
            signature=infer_signature(X_train[features], out),
        )
    return run.info.run_id, m


tests.test_run_experiment(run_experiment)
```

</details>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

run_experiment(RandomForestClassifier(n_estimators=300, max_depth=8, random_state=0), run_name="rf-depth-8")
run_experiment(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), run_name="logreg")
run_experiment(RandomForestClassifier(n_estimators=100, random_state=0), threshold=0.3, run_name="rf-threshold-0.3")
run_experiment(RandomForestClassifier(n_estimators=100, random_state=0),
               features=["type", "alcohol", "volatile acidity", "density"], run_name="rf-4-features")

In [ ]:
# The same table as the UI, as a DataFrame
runs = mlflow.search_runs(order_by=["metrics.balanced_accuracy DESC"])
cols = ["run_id", "tags.mlflow.runName", "params.threshold", "metrics.accuracy", "metrics.balanced_accuracy",
        "metrics.precision", "metrics.recall", "metrics.roc_auc"]
runs[cols]

In the UI, select the runs and press *Compare*. Then try a few of these, one run each:

- hyperparameters, another estimator, a `Pipeline` with scaling
- the decision threshold: what happens to precision and recall
- a subset of features: how much do you lose, what does the signature become
- a different target: `quality` as regression, which metrics make sense then
- a metric that matters: if a false positive costs 1 and a false negative costs 5, which run wins
- the split: another `random_state`, a smaller test set. Are the differences between runs larger than the noise?

## Part 6 · Registry

One name, many versions, an alias that points to the one in use. Promotion and rollback are
alias moves, not code changes.

Registering a run's model under a name (`wine-good`) gives it the next version number: 1, 2, ...
An alias such as `champion` is a pointer to one version, and can be moved. Code that uses the
model loads `models:/wine-good@champion` and never names a run or a version.
(Older MLflow used fixed "stages", Staging and Production; aliases replace them.)

In [ ]:
client = mlflow.MlflowClient()

chosen = runs.iloc[0]
mv = mlflow.register_model(f"runs:/{chosen.run_id}/model", "wine-good")
client.set_registered_model_alias("wine-good", "champion", mv.version)

champion = mlflow.pyfunc.load_model("models:/wine-good@champion")
print(champion.metadata.run_id == chosen.run_id)
champion.predict(X_test.head())

### Exercise · promote, then roll back

> Difficulty: 🔴⚪⚪⚪⚪ · Importance: 🔵🔵🔵⚪⚪ · about 10 minutes

A colleague prefers the logistic regression: simpler, and easier to explain. Register its model
as a new version of `wine-good` and point `champion` at it. Then compare its numbers with the
champion's in the table above, and put the previous version back. You need `mlflow.register_model` and `client.set_registered_model_alias`.
Watch the *Model registry* page of the UI as you go.

In [ ]:
logreg_id = runs.loc[runs["tags.mlflow.runName"] == "logreg", "run_id"].iloc[0]

# YOUR CODE HERE: register the logreg run's model under "wine-good", then point "champion" at the new version

tests.test_champion("wine-good", logreg_id)

<details><summary>Solution</summary>

```python
logreg_id = runs.loc[runs["tags.mlflow.runName"] == "logreg", "run_id"].iloc[0]

mv2 = mlflow.register_model(f"runs:/{logreg_id}/model", "wine-good")
client.set_registered_model_alias("wine-good", "champion", mv2.version)

tests.test_champion("wine-good", logreg_id)
```

</details>

In [ ]:
# YOUR CODE HERE: point "champion" back at the version you registered first (`mv`)

tests.test_champion("wine-good", chosen.run_id)

<details><summary>Solution</summary>

```python
client.set_registered_model_alias("wine-good", "champion", mv.version)

tests.test_champion("wine-good", chosen.run_id)
```

</details>

Nothing was retrained and no code was changed: the only thing that moved is a pointer.
Whatever loads `models:/wine-good@champion` gets the version it points to at load time.

## Part 7 · Serve

The same model behind an HTTP endpoint. The contract is the same, the payload is JSON.

`mlflow models serve` loads the model once, when it starts, and answers `POST /invocations`.
Moving the alias afterwards changes nothing until the server restarts. `--env-manager local` uses
the packages already installed in this VM. On a real server you would let MLflow build the
environment from the model's `requirements.txt` (`--env-manager virtualenv`), or package model and
server in a Docker image (`mlflow models build-docker`).

In [ ]:
# Plumbing: run and skip. Starts the model server in the background (takes up to a minute).
SERVE_PORT = 5001

serving = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "models", "serve", "-m", "models:/wine-good@champion", "-p", str(SERVE_PORT), "--env-manager", "local"],
    stdout=open("mlflow_serve.log", "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "MLFLOW_TRACKING_URI": TRACKING_URI,
         "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]},
)

for _ in range(120):
    try:
        if requests.get(f"http://127.0.0.1:{SERVE_PORT}/ping").ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("model server did not start, see mlflow_serve.log")

In [ ]:
INVOCATIONS = f"http://127.0.0.1:{SERVE_PORT}/invocations"
payload = {"dataframe_records": X_test.head(3).to_dict(orient="records")}
r = requests.post(INVOCATIONS, json=payload)
print(r.status_code)
r.json()

### Exercise · the broken rows, over HTTP

> Difficulty: 🔴⚪⚪⚪⚪ · Importance: 🔵🔵🔵⚪⚪ · about 10 minutes

Send each of the broken rows from part 4 to the endpoint. Compare status codes and messages with
the exceptions you got in Python. What does a caller who only sees the HTTP response learn?

In [ ]:
for name, df in cases.items():
    r = ...  # YOUR CODE HERE: post `df` to INVOCATIONS, in the same format as above
    assert r is not ..., "Fill in the blank above, then run this cell again."
    print(f"{name:22s} -> {r.status_code} {r.text[:160]}")

<details><summary>Solution</summary>

```python
for name, df in cases.items():
    r = requests.post(INVOCATIONS, json={"dataframe_records": df.to_dict(orient="records")})
    print(f"{name:22s} -> {r.status_code} {r.text[:160]}")
```

</details>

<details><summary>Answer</summary>

What came back on Colab:

- *missing column*: status 400, `INVALID_PARAMETER_VALUE`, and a long message that echoes the
  input rows back. Fine here; not with personal data in the rows.
- everything else, the text `"12.5"` included: status 200 and a prediction, as in Python.
  (With pandas 3, Python refuses `"12.5"` but the HTTP server still accepts it.)

A caller who sees status 200 and a number has no way to know the row was wrong. Whatever the
model lets through in Python it lets through over HTTP, and the JSON layer adds conversions of
its own. The checks have to live in the model.

</details>

## Cleanup

In [ ]:
serving.terminate()
server.terminate()